In [ ]:
import os

RAP_PROJECT_ID = os.environ["DNANEXUS_PROJECT_ID"]  # set your own DNAnexus RAP project ID


In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Configuration and paths
mac = 20

# ============================================================
# Variant class selection — change this to switch variant class
# Available: missense, missense_structured, missense_non_structured,
#   missense_disordered, missense_lip, intron,
#   enhancer_encode, promoter_encode, tf_encode, core_promoter
# ============================================================
variant_class = 'missense'
exclude_clinvar = False  # Independent toggle: exclude ClinVar-annotated variants

# Maximum number of top-ranked variants per annotation (None = no limit)
max_num_variants = None

# Initialize new annotation variable as None
new_anno_local = None

# Sliding window and bootstrap parameters
window_size = 1_000  # Number of variants in the sliding window
step_size = 1_0      # Step between sampled points (controls plot smoothness)
max_num_variants = 500_000

# LOEUF faceting
n_loeuf_bins = 4  # Number of LOEUF bins (default 4)
n_boot = 1000      # Number of bootstrap resamples across regions

# Load variant class configuration
variant_class_path = "PATH_TO_FILE"
with open(variant_class_path) as f:
    variant_class_config = yaml.safe_load(f)

vc = variant_class_config[variant_class]
vc_filters = vc['variant_filtering']
selected_categories = vc['tool_categories']
x_label = vc['x_label']

print(f"Variant class: {variant_class}")
print(f"  Filters: {vc_filters}")
print(f"  Categories: {selected_categories}")
print(f"  X-label: {x_label}")
print(f"  Exclude ClinVar: {exclude_clinvar}")

# Load annotation configuration
config_path = "PATH_TO_FILE"

with open(config_path) as f:
    config = yaml.safe_load(f)

eur_samples_path = 'PATH_TO_FILE'

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

In [ ]:
# Get gene trait associations
RAP_DIR = f'{RAP_PROJECT_ID}:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = 'PATH_TO_FILE'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
CORR_FILE = "regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.parquet"
!dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

loftee_corrs = (
    pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=["region"], keep="first", maintain_order=True)
)
gene_trait_df

In [ ]:
RAP_ANNO_DIR = f"{RAP_PROJECT_ID}:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "PATH_TO_FILE"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

# Build dynamic filters from variant_class.yaml
_dynamic_filters = [eval(f) for f in vc_filters]
if exclude_clinvar:
    _dynamic_filters.append(pl.col('clinical_significance').is_null())

anno = (
    anno
    .filter(
        # Always-applied filters
        (pl.col('region').is_in(gene_trait_df['region'].unique())),
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1),

        # Dynamic filters from variant_class.yaml + exclude_clinvar
        *_dynamic_filters,
    )
)

# selected_categories is already set from YAML in the config cell

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
fillna_cols = [c+'_is_na' for c in selected_annos]

anno = (
    anno
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

In [ ]:
anno_fillna = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
fillna_cols = set([c+'_is_na' for c in selected_annos]).intersection(set(anno_fillna.collect_schema().names()))

anno_fillna_melted = (
    anno_fillna
    .filter(pl.col('region').is_in(gene_trait_df['region'].unique()))
    .select(['id', 'region'] + list(fillna_cols))
    .join(
        anno.select(['id', 'region']).lazy(), 
        on=['id', 'region'], 
        how='semi'
    )
    .unpivot(
        index=["id", "region"],
        on=list(fillna_cols),
        variable_name="annotation",
        value_name="annotation_is_na"
    )
    .filter(
        pl.col('annotation_is_na') == 1
    )
    .with_columns(
        annotation = pl.col('annotation').str.replace('_is_na$', '')
    )
)

anno_fillna_melted.head().collect()

In [ ]:
melted_anno = (
    anno.lazy()

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df.select(["annotation", "category", "annotation_dir"]).lazy(),
        on="annotation",
        how="left"
    )
    .filter(pl.col('category').is_in(selected_categories))
    .unique()

    # Remove null annotation scores
    .drop_nulls('annotation_score')

    # Remove variants for each annotation which are not scored
    .join(
        anno_fillna_melted,
        on=['id', 'region', 'annotation'],
        how='anti'
    )

    # Correct scores by annotation direction
    .with_columns(
        annotation_score_dircor = pl.col('annotation_score') * pl.col("annotation_dir").cast(pl.Float32)
    )
    
    # Implement filtering based on the defined percentile
    .with_columns(
        annotation_score_dircor_rank_desc = pl.col('annotation_score_dircor').rank(method="max", descending=True).over(["annotation"]).cast(pl.Float32)
    )
    .drop(['annotation_score_dircor'])

    .collect(engine='streaming')
)

melted_anno

In [ ]:
melted_anno['annotation'].value_counts(sort=True)

In [ ]:
RAP_APPV_DIR = f"{RAP_PROJECT_ID}:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "PATH_TO_FILE"

APPV_FILE = "quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
pheno_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Filter to variants in annotation set and low MAC
anno_keys = anno.select(pl.col('id').unique()).lazy()

pheno_appv = (
    pheno_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
)

In [ ]:
pheno_appv.head().collect()

## Average z-score by annotation rank (sliding window)

In [ ]:
# Load gnomAD constraint metrics and create LOEUF bins
gnom = (
    pl.read_csv(
        "PATH_TO_FILE", 
        null_values=["NA"],
        separator='\t'
    )
    .filter(
        pl.col('transcript_type')=='protein_coding',
        pl.col('canonical')==True
    )
    .select(['gene_id', 'lof.oe_ci.upper'])
    .rename({'gene_id': 'region', 'lof.oe_ci.upper': 'loeuf'})
    .drop_nulls()
)

loeuf_labels = [
    f'Q{i+1} (most constrained)' if i == 0
    else f'Q{i+1} (least constrained)' if i == n_loeuf_bins - 1
    else f'Q{i+1}'
    for i in range(n_loeuf_bins)
]

loeuf_bins = gnom.with_columns(
    loeuf_bin = pl.col('loeuf').qcut(n_loeuf_bins, labels=loeuf_labels)
)

loeuf_bins

In [ ]:
# --- Helper functions ---

def sliding_window_means_unweighted(zscores, bin_ends, window_size):
    """Standard prefix-sum sliding window mean (for point estimates)."""
    csum = np.empty(len(zscores) + 1, dtype=np.float64)
    csum[0] = 0.0
    np.cumsum(zscores, out=csum[1:])
    return (csum[bin_ends] - csum[bin_ends - window_size]) / window_size


def expanded_csum_at(positions, cw, cwz, z, N):
    """Compute the prefix sum of the virtual expanded array at given positions."""
    k = np.searchsorted(cw, positions, side='left')
    k_safe = np.clip(k, 1, N)
    result = cwz[k_safe - 1] + z[k_safe - 1] * (positions - cw[k_safe - 1])
    result = np.where(positions <= 0, 0.0, result)
    return result


def sliding_window_reranked(z, cw, cwz, N, total_expanded, bin_ends, window_size):
    """Sliding window mean on the virtual expanded (re-ranked) array."""
    valid_mask = bin_ends <= total_expanded
    valid_bins = bin_ends[valid_mask]
    means = np.full(len(bin_ends), np.nan, dtype=np.float64)
    if len(valid_bins) == 0:
        return means
    right = expanded_csum_at(valid_bins, cw, cwz, z, N)
    left = expanded_csum_at(valid_bins - window_size, cw, cwz, z, N)
    means[valid_mask] = (right - left) / window_size
    return means


# --- Join gene_trait_df with LOEUF bins ---
gene_trait_loeuf = gene_trait_df.join(loeuf_bins, on='region', how='inner')

bin_counts = (
    gene_trait_loeuf
    .select(['region', 'loeuf_bin'])
    .unique(subset=['region'])
    .group_by('loeuf_bin')
    .len()
)
bin_label_map = {row[0]: f"{row[0]} (n={row[1]})" for row in bin_counts.iter_rows()}

id_region = anno.select(['id', 'region']).unique().lazy()

# --- Loop over LOEUF bins ---
all_zscore_binned = []

for loeuf_bin_raw, loeuf_bin_label in sorted(bin_label_map.items()):
    print(f"\n{'='*60}")
    print(f"Processing: {loeuf_bin_label}")
    print(f"{'='*60}")

    # Filter gene_trait_df to this bin
    gt_bin = gene_trait_loeuf.filter(pl.col('loeuf_bin') == loeuf_bin_raw)
    bin_regions = gt_bin['region'].unique()

    # Filter melted_anno to this bin's regions and re-rank within the subset
    melted_anno_bin = (
        melted_anno
        .filter(pl.col('region').is_in(bin_regions))
        .with_columns(
            annotation_score_dircor = pl.col('annotation_score') * pl.col('annotation_dir').cast(pl.Float32)
        )
        .with_columns(
            annotation_score_dircor_rank_desc = pl.col('annotation_score_dircor')
                .rank(method="max", descending=True)
                .over("annotation")
                .cast(pl.Float32)
        )
        .drop('annotation_score_dircor')
    )

    # Variant z-scores for this bin
    variant_zscores = (
        pheno_appv
        .join(id_region, on='id', how='inner')
        .join(gt_bin.lazy(), on=['region', 'phenotype'], how='inner')
        .with_columns(
            mean_pheno_value = pl.col('mean_pheno_value') * pl.col('loftee_corr_dir').cast(pl.Float32)
        )
        .select(['id', 'region', 'mean_pheno_value'])
        .collect(engine='streaming')
    )
    print(f"  Variant z-scores: {variant_zscores.shape[0]} variant-gene pairs")

    # Ranked z-scores
    ranked_zscores = (
        variant_zscores.lazy()
        .join(
            melted_anno_bin.lazy().select(['id', 'region', 'annotation', 'annotation_score_dircor_rank_desc']),
            on=['id', 'region'],
            how='inner'
        )
        .sort(['annotation', 'annotation_score_dircor_rank_desc'])
        .with_columns(
            row_pos = pl.col('annotation').cum_count().over('annotation'),
        )
        .pipe(lambda df: df.filter(pl.col('row_pos') <= max_num_variants) if max_num_variants is not None else df)
        .collect(engine='streaming')
    )

    # Map regions to integer indices for fast bootstrap resampling
    bin_all_regions = sorted(gt_bin['region'].unique().to_list())
    bin_n_regions = len(bin_all_regions)
    region_idx_map = pl.DataFrame({'region': bin_all_regions, '_region_idx': np.arange(bin_n_regions, dtype=np.int32)})
    ranked_zscores = ranked_zscores.join(region_idx_map, on='region', how='left')

    annotations_list = sorted(ranked_zscores['annotation'].unique().to_list())

    # --- Prepare per-annotation sorted arrays and bin endpoints ---
    anno_arrays = {}
    for annotation in tqdm(annotations_list, desc=f'  Preparing arrays'):
        adf = ranked_zscores.filter(pl.col('annotation') == annotation)
        z = adf['mean_pheno_value'].to_numpy().astype(np.float64)
        r_idx = adf['_region_idx'].to_numpy()
        N = len(z)
        bin_ends = np.arange(window_size, N + 1, step_size, dtype=np.int64)
        anno_arrays[annotation] = {
            'zscores': z,
            'region_idx': r_idx,
            'bin_ends': bin_ends,
            'N': N,
        }

    # --- Point estimates (unweighted, on full data) ---
    point_means = {}
    for annotation in annotations_list:
        d = anno_arrays[annotation]
        point_means[annotation] = sliding_window_means_unweighted(
            d['zscores'], d['bin_ends'], window_size
        )

    # --- Re-ranking bootstrap CIs across regions ---
    rng = np.random.default_rng(42)
    boot_means = {ann: np.full((n_boot, len(anno_arrays[ann]['bin_ends'])), np.nan)
                  for ann in annotations_list}

    max_N = max(d['N'] for d in anno_arrays.values())
    cw_buf = np.empty(max_N + 1, dtype=np.float64)
    cwz_buf = np.empty(max_N + 1, dtype=np.float64)

    for b in tqdm(range(n_boot), desc=f'  Bootstrap'):
        idx = rng.choice(bin_n_regions, size=bin_n_regions, replace=True)
        region_weights = np.bincount(idx, minlength=bin_n_regions).astype(np.float64)

        for annotation in annotations_list:
            d = anno_arrays[annotation]
            N = d['N']
            z = d['zscores']
            vw = region_weights[d['region_idx']]

            cw = cw_buf[:N + 1]
            cw[0] = 0.0
            np.cumsum(vw, out=cw[1:])

            cwz = cwz_buf[:N + 1]
            cwz[0] = 0.0
            np.cumsum(z * vw, out=cwz[1:])

            total_expanded = int(cw[N])

            boot_means[annotation][b] = sliding_window_reranked(
                z, cw, cwz, N, total_expanded, d['bin_ends'], window_size
            )

    # --- Build output DataFrame with percentile-based CIs ---
    bin_results = []
    for annotation in annotations_list:
        d = anno_arrays[annotation]
        bm = boot_means[annotation]
        bin_results.append(pl.DataFrame({
            'annotation': annotation,
            'bin_end': d['bin_ends'].astype(np.float64),
            'mean_zscore': point_means[annotation].astype(np.float32),
            'ci_lower': np.nanpercentile(bm, 2.5, axis=0).astype(np.float32),
            'ci_upper': np.nanpercentile(bm, 97.5, axis=0).astype(np.float32),
        }))

    zscore_binned_bin = (
        pl.concat(bin_results)
        .sort(['annotation', 'bin_end'])
        .with_columns(loeuf_bin=pl.lit(loeuf_bin_label))
    )
    print(f"  Sliding window results: {zscore_binned_bin.shape[0]} points across {zscore_binned_bin['annotation'].n_unique()} annotations")
    all_zscore_binned.append(zscore_binned_bin)

zscore_binned = pl.concat(all_zscore_binned).sort(['loeuf_bin', 'annotation', 'bin_end'])
print(f"\nTotal: {zscore_binned.shape[0]} points across {zscore_binned['loeuf_bin'].n_unique()} LOEUF bins")
zscore_binned

## Plotting

In [ ]:
plt_df = (
    zscore_binned
    .drop_nans()
    .with_columns(
        log_bin_end = pl.col('bin_end').log10(),
        log_bin_end_inv = (1 / pl.col('bin_end')).log10(),
    )
    .join(
        anno_config_df.filter(pl.col('category').is_in(selected_categories)).select(['annotation', 'color', 'label', 'category']).unique(),
        on='annotation',
        how='left'
    )
    .filter(pl.col('category').is_in(selected_categories))
    .sort('bin_end')
)

plt_df

In [ ]:
plot_df = (
    plt_df
    .filter(
        (pl.col('bin_end') <= 10_000),
        
        # (pl.col('annotation').is_in(['promoterai', 'fz_all_min']))
        # (pl.col('annotation').is_in(['promoterai', 'fz_all_max']))
        (~pl.col('annotation').is_in(['esm1b_llr', 'esmscoremissense']))
    )
)

# Generate breaks and labels for x-axis (most extreme on the left)
max_rank = zscore_binned['bin_end'].max()
min_rank = zscore_binned['bin_end'].min()
start_exp = int(np.floor(np.log10(min_rank)))
end_exp = int(np.ceil(np.log10(max_rank)))
breaks_linear = np.arange(start_exp, end_exp + 1)
labels_sci = [f"$10^{{{int(b)}}}$" for b in breaks_linear]

color_dict = dict(zip(plt_df['label'], plt_df['color']))

(
    ggplot(
        plot_df,
        # aes(x='log_bin_end', y='mean_zscore')
        aes(x='bin_end', y='mean_zscore')
    )
    # + loftee_point
    + geom_hline(aes(yintercept=0), color='black', linetype='dotted')
    + geom_line(aes(color='label'))
    + geom_ribbon(aes(ymin='ci_lower', ymax='ci_upper', fill='label'), alpha=0.1)
    + facet_wrap("~loeuf_bin", ncol=2)
    + scale_fill_manual(values=color_dict)
    + scale_color_manual(values=color_dict)
    + labs(
        title=f"Olink z-scores stratified by LOEUF quartile\nsliding window of {window_size} variants ({n_boot} bootstrap resamples)",
        y="Mean Olink z-score",
        x=x_label,
        color="Annotation",
        fill="Annotation",
    )
    # + scale_x_continuous(
    #     breaks=breaks_linear,
    #     labels=labels_sci
    # )
    + theme_minimal()
    + theme(
        figure_size=(10, 8),
        title=element_text(size=13, lineheight=1.4),
        axis_text=element_text(size=11),
        axis_title=element_text(size=13, lineheight=1.4),
        legend_text=element_text(size=11),
        legend_title=element_text(size=12),
        strip_text=element_text(size=12),
        # legend_position='bottom',
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [ ]:
# Point plot: mean z-score of top window_size variants per annotation, faceted by LOEUF bin
point_df = (
    plot_df
    .filter(pl.col('bin_end') == float(window_size))
)

# Order annotations by mean z-score (averaged across LOEUF bins), descending
label_order = (
    point_df
    .group_by('label')
    .agg(pl.col('mean_zscore').mean())
    .sort('mean_zscore', descending=False)
    ['label'].to_list()
)

color_dict = dict(zip(plt_df['label'], plt_df['color']))

(
    ggplot(
        point_df,
        aes(x='label', y='mean_zscore', color='label')
    )
    # + geom_hline(aes(yintercept=0), color='grey', linetype='dotted')
    + geom_point(size=3)
    + geom_errorbar(aes(ymin='ci_lower', ymax='ci_upper'), width=0.2)
    + facet_wrap("~loeuf_bin", ncol=2)
    + scale_color_manual(values=color_dict)
    + scale_x_discrete(limits=label_order)
    # + coord_flip()
    + labs(
        title=f"Top {window_size} variants per annotation\nstratified by LOEUF quartile ({n_boot} bootstrap resamples)",
        y="Mean Olink z-score",
        x="",
        fill="Annotation",
        color="Annotation",
    )
    + guides(color=None)
    + theme_minimal()
    + theme(
        figure_size=(8, 8),
        title=element_text(size=13, lineheight=1.4),
        axis_text=element_text(size=11),
        axis_text_x=element_text(size=0),
        axis_title=element_text(size=13),
        strip_text=element_text(size=12),
        plot_background=element_rect(fill="white", color="white"),
    )
)